# Fractured X-ray Image Augmentation

This notebook augments fractured X-ray images to address class imbalance, then organizes
everything into the YOLO-compatible dataset structure.

## Problem
- Fractured images: 717
- Non-fractured images: 3366
- Ratio: ~1:4.7 (highly imbalanced)

## Solution
Apply realistic X-ray augmentations to **training-set fractured images only**, saving both
the augmented images and their transformed YOLO bounding-box labels.

## Steps
1. Set up paths and count images
2. Preview augmentations on sample images
3. Run augmentation (images + labels)
4. Check class balance
5. Run `organize_dataset.py` to build the final YOLO folder structure

## 1. Imports & Setup

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path
from tqdm.notebook import tqdm

print(f"albumentations version: {A.__version__}")

In [ ]:
# --- Paths ---
BASE_DIR          = Path.cwd()
DATA_DIR          = BASE_DIR / "datasets"
IMAGES_DIR        = DATA_DIR / "images"
FRACTURED_DIR     = IMAGES_DIR / "Fractured"
NON_FRACTURED_DIR = IMAGES_DIR / "Non_fractured"
FRACTURED_AUG_DIR = IMAGES_DIR / "Fractured_Aug"
LABELS_DIR        = DATA_DIR / "labels" / "Fractured_Aug"
YOLO_LABELS_DIR   = BASE_DIR / ".." / "FracAtlas" / "Annotations" / "YOLO"
DISTRIBUTION_DIR  = BASE_DIR / "Distribution"

AUGMENTATIONS_PER_IMAGE = 3

# Create output folders
FRACTURED_AUG_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

# Count current images
def count_images(folder):
    return sum(1 for _ in folder.glob("*.jpg")) + sum(1 for _ in folder.glob("*.png"))

fractured_count     = count_images(FRACTURED_DIR)
non_fractured_count = count_images(NON_FRACTURED_DIR)

print(f"Fractured images     : {fractured_count}")
print(f"Non-fractured images : {non_fractured_count}")
print(f"Imbalance ratio      : {non_fractured_count / fractured_count:.2f}:1")

## 2. Load Training Split

In [ ]:
def load_csv_ids(csv_path: Path) -> set:
    """Read image filenames from a Distribution CSV."""
    ids = set()
    with open(csv_path) as f:
        for line in f:
            line = line.strip()
            if not line or line == "image_id":
                continue
            ids.add(line)
    return ids

train_ids = load_csv_ids(DISTRIBUTION_DIR / "train.csv")

# Only augment fractured images that are in the training split
training_fractured = [
    FRACTURED_DIR / name
    for name in train_ids
    if (FRACTURED_DIR / name).exists()
]

print(f"Total training IDs        : {len(train_ids)}")
print(f"Fractured in training set : {len(training_fractured)}")
print(f"Will create               : {len(training_fractured) * AUGMENTATIONS_PER_IMAGE} augmented images")

## 3. Augmentation Pipeline

All transforms are physically plausible for X-ray images:

- **Geometric** — flips, rotations, affine shifts (patient positioning variance)
- **Intensity** — brightness/contrast (exposure differences between machines)
- **Noise** — Gaussian noise (sensor/quantum mottle)
- **Blur** — Gaussian blur (patient motion, slight defocus)
- **Elastic** — mild tissue/bone deformation
- **Gamma / CLAHE** — exposure correction and local contrast enhancement

Bounding boxes are transformed alongside images so YOLO labels stay accurate.

In [ ]:
xray_augmentation = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=25, p=0.5),
    A.Affine(translate_percent=(-0.05, 0.05), scale=(0.90, 1.10), rotate=(-15, 15), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.35),
    A.GaussianBlur(blur_limit=(1, 5), p=0.2),
    A.ElasticTransform(alpha=1.0, sigma=20, p=0.15),
    A.RandomGamma(gamma_limit=(85, 115), p=0.35),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.25),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print("Augmentation pipeline ready.")

## 4. Preview on Sample Images

In [ ]:
sample_paths = training_fractured[:3]

fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for i, img_path in enumerate(sample_paths):
    original = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    augmented = xray_augmentation(image=original, bboxes=[], class_labels=[])
    aug_image = augmented['image']

    axes[i, 0].imshow(original)
    axes[i, 0].set_title(f"Original: {img_path.name}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(aug_image)
    axes[i, 1].set_title("Augmented")
    axes[i, 1].axis('off')

plt.suptitle("Sample Augmentation Preview", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Run Augmentation (Images + YOLO Labels)

In [ ]:
def parse_yolo_label(label_path: Path):
    """Parse a YOLO .txt label file → list of [x, y, w, h, class_id]."""
    bboxes = []
    if not label_path.exists():
        return bboxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:5])
                bboxes.append([x, y, w, h, cls])
    return bboxes


def write_yolo_label(label_path: Path, bboxes):
    """Write bboxes [[x, y, w, h, class_id], ...] to a YOLO .txt file."""
    with open(label_path, 'w') as f:
        for b in bboxes:
            x, y, w, h, cls = b
            f.write(f"{int(cls)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


augmented_count = 0
total_to_create = len(training_fractured) * AUGMENTATIONS_PER_IMAGE

for img_path in tqdm(training_fractured, desc="Augmenting"):
    image = cv2.imread(str(img_path))
    if image is None:
        print(f"  Warning: could not load {img_path.name}")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Load YOLO label for this image
    label_path = YOLO_LABELS_DIR / f"{img_path.stem}.txt"
    bboxes     = parse_yolo_label(label_path)

    for aug_idx in range(AUGMENTATIONS_PER_IMAGE):
        result = xray_augmentation(
            image=image,
            bboxes=[[b[0], b[1], b[2], b[3]] for b in bboxes],
            class_labels=[b[4] for b in bboxes],
        )

        aug_stem = f"{img_path.stem}_aug{aug_idx + 1:03d}"

        # Save image
        aug_img_bgr = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
        cv2.imwrite(
            str(FRACTURED_AUG_DIR / f"{aug_stem}.jpg"),
            aug_img_bgr,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        # Save label (re-pair surviving bboxes with their class ids)
        aug_bboxes = [
            list(b) + [c]
            for b, c in zip(result['bboxes'], result['class_labels'])
        ]
        write_yolo_label(LABELS_DIR / f"{aug_stem}.txt", aug_bboxes)

        augmented_count += 1

print(f"\nDone! Created {augmented_count} augmented images and {augmented_count} label files.")

## 6. Class Balance Analysis

In [ ]:
actual_augmented = count_images(FRACTURED_AUG_DIR)
all_fractured    = fractured_count + actual_augmented

print(f"Original fractured  : {fractured_count}")
print(f"Augmented fractured : {actual_augmented}")
print(f"Total fractured     : {all_fractured}")
print(f"Non-fractured       : {non_fractured_count}")
print(f"Old ratio  : {non_fractured_count / fractured_count:.2f}:1")
print(f"New ratio  : {non_fractured_count / all_fractured:.2f}:1")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
categories = ['Non-fractured', 'Fractured\n(Original)', 'Fractured\n(Augmented)']
counts     = [non_fractured_count, fractured_count, actual_augmented]
colors     = ['steelblue', 'tomato', 'salmon']
bars = ax.bar(categories, counts, color=colors, edgecolor='black')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{count:,}', ha='center', va='bottom', fontweight='bold')

ax.set_title('X-ray Image Class Distribution After Augmentation', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Images')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.text(0.5, 0.95,
        f'New ratio: {non_fractured_count / all_fractured:.2f}:1  (was {non_fractured_count / fractured_count:.2f}:1)',
        transform=ax.transAxes, ha='center', va='top', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.8))
plt.tight_layout()
plt.show()

## 7. Organize Dataset (YOLO Structure)
Runs `organize_dataset.py`, which picks up the augmented images and copies everything
into `datasets/dataset/train|valid|test/images+labels/`.

In [ ]:
!python ../scripts/organize_dataset.py

## Done ✓

The pipeline is complete:
- Augmented images saved to `datasets/images/Fractured_Aug/`
- Transformed YOLO labels saved to `datasets/labels/Fractured_Aug/`
- Full YOLO dataset organized under `datasets/dataset/`

You can now run `Train_8s.ipynb` for YOLO detection training, or the
`KNN_Classifier_Augmented.ipynb` / `SVM_Classifier_Augmented.ipynb` notebooks
for classical ML classification.